In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ==========================================================
# Configuration
# ==========================================================

BRONZE_TABLE = "finance_intelligence_hub.bronze.stocks_info"
SILVER_TABLE = "finance_intelligence_hub.silver.stocks_info"

print("=" * 70)
print("Finance Intelligence Hub")
print("Silver Layer Pipeline - Stocks Info")
print("=" * 70)

# ==========================================================
# Create Silver Table if it does not exist
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {SILVER_TABLE}

(
ticker STRING,
symbol STRING,
company_name STRING,
short_name STRING,
exchange STRING,
currency STRING,
country STRING,
sector STRING,
industry STRING,
market_cap DOUBLE,
shares_outstanding DOUBLE,
quote_type STRING,
dwh_loaded_at TIMESTAMP,
silver_loaded_at TIMESTAMP
)

USING DELTA

""")

print("Silver table verified.")

# ==========================================================
# Incremental Load
# ==========================================================

last_loaded = spark.sql(
    f"""SELECT COALESCE(MAX(dwh_loaded_at),TIMESTAMP('1900-01-01')) AS last_load FROM {SILVER_TABLE}"""
).first()["last_load"]

print(f"Last Silver Load : {last_loaded}")

bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("dwh_loaded_at") > F.lit(last_loaded))
)

rows = bronze_df.count()
print(f"New Bronze Records : {rows}")

if rows == 0:
    print("No new records found.")
    dbutils.notebook.exit("Silver layer already up to date.")

# ==========================================================
# Null Handling
# ==========================================================

clean_df = (
    bronze_df
    .withColumn(
        "company_name",
        F.coalesce(F.col("company_name"), F.col("short_name"), F.lit("Unknown"))
    )
    .withColumn(
        "short_name",
        F.coalesce(F.col("short_name"), F.col("company_name"), F.lit("Unknown"))
    )
    .withColumn("currency", F.coalesce(F.col("currency"), F.lit("USD")))
    .withColumn("country", F.coalesce(F.col("country"), F.lit("Unknown")))
    .withColumn("sector", F.coalesce(F.col("sector"), F.lit("Unknown")))
    .withColumn("industry", F.coalesce(F.col("industry"), F.lit("Unknown")))
    .withColumn("silver_loaded_at", F.current_timestamp())
)

silver_df = clean_df.select(
    "ticker", "symbol", "company_name", "short_name", "exchange",
    "currency", "country", "sector", "industry",
    "market_cap", "shares_outstanding", "quote_type",
    "dwh_loaded_at", "silver_loaded_at"
)

print("=" * 70)
print("Null Handling Completed")
print("=" * 70)

# ==========================================================
# MERGE INTO Silver Layer
# ==========================================================

print("=" * 70)
print("Starting MERGE into Silver Table...")
print("=" * 70)

compare_cols = [
    "symbol", "company_name", "short_name", "exchange",
    "currency", "country", "sector", "industry",
    "market_cap", "shares_outstanding", "quote_type"
]

change_condition = " OR ".join(
    f"COALESCE(CAST(source.{c} AS STRING), '') <> COALESCE(CAST(target.{c} AS STRING), '')"
    for c in compare_cols
)

delta_table = DeltaTable.forName(spark, SILVER_TABLE)

(
    delta_table.alias("target")
    .merge(
        silver_df.alias("source"),
        "target.ticker = source.ticker"
    )
    .whenMatchedUpdate(
        condition=f"source.dwh_loaded_at > target.dwh_loaded_at AND ({change_condition})",
        set={
            "symbol": "source.symbol",
            "company_name": "source.company_name",
            "short_name": "source.short_name",
            "exchange": "source.exchange",
            "currency": "source.currency",
            "country": "source.country",
            "sector": "source.sector",
            "industry": "source.industry",
            "market_cap": "source.market_cap",
            "shares_outstanding": "source.shares_outstanding",
            "quote_type": "source.quote_type",
            "dwh_loaded_at": "source.dwh_loaded_at",
            "silver_loaded_at": "source.silver_loaded_at"
        }
    )
    .whenNotMatchedInsert(
        values={
            "ticker": "source.ticker",
            "symbol": "source.symbol",
            "company_name": "source.company_name",
            "short_name": "source.short_name",
            "exchange": "source.exchange",
            "currency": "source.currency",
            "country": "source.country",
            "sector": "source.sector",
            "industry": "source.industry",
            "market_cap": "source.market_cap",
            "shares_outstanding": "source.shares_outstanding",
            "quote_type": "source.quote_type",
            "dwh_loaded_at": "source.dwh_loaded_at",
            "silver_loaded_at": "source.silver_loaded_at"
        }
    )
    .execute()
)

print("MERGE completed successfully.")

spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (ticker)")

total_rows = spark.sql(f"SELECT COUNT(*) FROM {SILVER_TABLE}").first()[0]
print(f"\nSilver Pipeline Completed. Total Records : {total_rows:,}")